# Bangel Fish Detection with YOLOv11
 
 **Description:**  
This notebook trains, validates, and evaluates a YOLOv11 object detector on the Bangel Fish dataset using Yolov11 approach.
 
**Sections:**  
1. 📦 Environment setup & imports  
2. 🗂️ Dataset Load from Roboflow 
3. ⚙️ Data configuration  
4. 🚀 Model training  
5. 🔍 Validation & test evaluation  
 
**Environment:** Kaggle  
**Date:** 2025-07-22

# Load Dataset and Libraries


In [ ]:
!pip install roboflow[]

In [ ]:
%pip install ultralytics supervision roboflow
import ultralytics
ultralytics.checks()

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="nQzhXNDmyUoTO8vH1s8O")
project = rf.workspace("fish-detection-dgju1").project("bd-delta-fish-ygdsq")
version = project.version(1)
dataset = version.download("yolov11")

provides tools for training, validating, and deploying YOLO 

In [ ]:
!pip install ultralytics

In [ ]:
import os

print("data.yaml exists:", os.path.exists("/kaggle/working/-bd-delta-fish--1/data.yaml"))

In [ ]:
print("train/images exists:", os.path.exists("/kaggle/working/-bd-delta-fish--1/train/images"))
print("valid/images exists:", os.path.exists("/kaggle/working/-bd-delta-fish--1/valid/images"))
print("test/images exists:", os.path.exists("/kaggle/working/-bd-delta-fish--1/test/images"))

In [ ]:
with open(f"{dataset.location}/data.yaml", "r") as f:
    print(f.read())

In [ ]:
import glob

base_path = "/kaggle/working/-bd-delta-fish--1"

for folder in ["train", "valid", "test"]:
    image_path = os.path.join(base_path, folder, "images")
    images = glob.glob(os.path.join(image_path, "*"))

    print(f"{folder} images: {len(images)}")

# Data Preparation

In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob
import os

# YOLO label loader
def load_yolo_labels(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            cls, x, y, w, h = map(float, line.strip().split())
            boxes.append((int(cls), x, y, w, h))
    return boxes

# Image and label paths
image_paths = glob.glob(os.path.join(dataset.location, "train", "images", "*.jpg"))[:20]

# Class names from YAML
class_names = ['Batashi', 'Bele', 'Chanda', 'Chapila', 'Chela', 'Foli', 'Ilish', 'Kajuli', 'Kayakanta', 'Koi', 
               'Magur', 'Mola', 'Mrigel', 'Pabda', 'Pangas', 'Puti', 'Rui', 'Rupchada', 'Shol', 'Sorputi', 'Taki',
               'Tarabaim', 'Telapia', 'Tengra', 'Veda', 'Vetki', 'ayer', 'boyal', 'chitol', 'kalbaus', 'katol', 
               'kholshe', 'shing']

# Plotting
for img_path in image_paths:
    label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    boxes = load_yolo_labels(label_path)

    for cls, x, y, bw, bh in boxes:
        x1 = int((x - bw / 2) * w)
        y1 = int((y - bh / 2) * h)
        x2 = int((x + bw / 2) * w)
        y2 = int((y + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 4)
        cv2.putText(img, class_names[cls], (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 5, (255, 0, 0), 9)

    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Train yolo11s model training


In [ ]:
from ultralytics import YOLO
import time

# Load the pre-trained YOLOv11s model
model = YOLO("yolo11s.pt") 

# Start timer
start_time = time.time()

# Train the model on the custom fish dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",  
    epochs=50,
    imgsz=640,
    batch=16,
    name="fish-yolo11s",
    verbose=True,
    device=0  
)

# End timer
end_time = time.time()
training_time = end_time - start_time

# Print training summary
print("\n✅ Training Complete!")
print(f"⏱️ Total training time: {training_time:.2f} seconds")


# Model Evaluation

In [ ]:
# Training time print
print("\n✅ Training Complete!")
print(f"⏱️ Total training time: {training_time:.2f} seconds")

# Directly access metrics from 'results' (because results is already a DetMetrics object)
if hasattr(results, "box") and results.box.map is not None:
    print(f"📊 Best mAP@[.5:.95]: {results.box.map:.4f}")
else:
    print("📊 mAP@[.5:.95] not available")

# FPS from inference speed
if hasattr(results, "speed") and "inference" in results.speed:
    fps = 1000 / results.speed["inference"]
    print(f"⚡ Inference FPS: {fps:.2f}")
else:
    print("⚡ FPS data not available")


In [ ]:
print(os.listdir("/kaggle/working/runs/detect"))

## Training & Validation Loss Curve - YOLO11s

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "/kaggle/working/runs/detect/fish-yolo11s/results.csv"

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

plt.figure(figsize=(12, 6))

plt.plot(df["epoch"], df["train/box_loss"], label="Train Box Loss")
plt.plot(df["epoch"], df["val/box_loss"], label="Validation Box Loss")

plt.plot(df["epoch"], df["train/cls_loss"], label="Train Class Loss")
plt.plot(df["epoch"], df["val/cls_loss"], label="Validation Class Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss Curve - YOLO11s")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

base_dir = "runs/detect"
for folder in os.listdir(base_dir):
    print(folder)

In [ ]:
import glob

best_weight_path = results.save_dir / "weights" / "best.pt"

print("Best Weight Path:", best_weight_path)


In [ ]:
from ultralytics import YOLO

model = YOLO(str(best_weight_path))
val_results = model.val(
    data="/kaggle/working/-bd-delta-fish--1/data.yaml",
    split="val",
    save=True
)



In [ ]:
from ultralytics import YOLO
import time

model = YOLO(best_weight_path)
start_time = time.time()

# ✅ Step 4: Evaluate on the test set
results = model.val(
    data=f"{dataset.location}/data.yaml",  
    split='test',
    device=0,
save=True,
plots=True
)

# End timer
end_time = time.time()
eval_time = end_time - start_time

# 📊 Evaluation Summary
print("\n📊 Evaluation Results on Test Set:")
if hasattr(results, "box"):
    print(f"🔹 Overall mAP@[.5:.95]: {results.box.map:.4f}")
    print(f"🔹 mAP@0.5: {results.box.map50:.4f}")
    print(f"🔹 mAP@0.75: {results.box.map75:.4f}")

In [ ]:
print("Saved results at:", results.save_dir)
image_paths = glob.glob(
    os.path.join(str(results.save_dir), "*.jpg")
)

print("Total images:", len(image_paths))

In [ ]:
print("\n📋 Per-class AP:")
per_class_aps = results.box.maps 

for cls_name, ap in zip(results.names.values(), per_class_aps):
    print(f"{cls_name:15}: AP = {ap:.4f}")

## Top and Bottom 3 class by map:

In [ ]:
per_class_aps = results.box.maps  
class_names = list(results.names.values())

top3_indices = sorted(range(len(per_class_aps)), key=lambda i: per_class_aps[i], reverse=True)[:3]

top3_class_names = [class_names[i] for i in top3_indices]

print("\n Top-3 Classes by mAP:")
for i in top3_indices:
    print(f"{class_names[i]:15}: AP = {per_class_aps[i]:.4f}")
    # Bottom-3 classes
bottom3_indices = sorted(
    range(len(per_class_aps)),
    key=lambda i: per_class_aps[i]
)[:3]

print("\n🔴 Bottom-3 Classes by mAP:")

for i in bottom3_indices:
    print(f"{class_names[i]:15}: AP = {per_class_aps[i]:.4f}")

In [ ]:
import glob
import os

test_images = glob.glob(
    "/kaggle/working/-bd-delta-fish--1/test/images/*.jpg"
)

print("Total test images:", len(test_images))

## Top-3 Classes by mAP:


In [ ]:
# 🟢 Top-3 Classes by mAP

for cls_id in top3_indices:

    print(f"\nClass: {class_names[cls_id]}")
    print(f"AP: {per_class_aps[cls_id]:.4f}")

    for img_path in test_images:

        label_path = img_path.replace(
            os.sep + "images" + os.sep,
            os.sep + "labels" + os.sep
        )
        label_path = os.path.splitext(label_path)[0] + ".txt"

        if not os.path.exists(label_path):
            continue

        with open(label_path, "r") as f:
            labels = [
                list(map(float, line.split()))
                for line in f if line.strip()
            ]

        if not any(int(label[0]) == cls_id for label in labels):
            continue

        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        # Ground Truth
        gt_img = img.copy()

        for label in labels:
            cls, x, y, bw, bh = label

            if int(cls) != cls_id:
                continue

            x1 = int((x - bw/2) * w)
            y1 = int((y - bh/2) * h)
            x2 = int((x + bw/2) * w)
            y2 = int((y + bh/2) * h)

            cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 12)

            cv2.putText(
                gt_img, class_names[cls_id],
                (x1, max(y1-15, 35)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.2, (0, 255, 0), 4
            )

        # YOLO Prediction
        pred_result = model.predict(
            img_path, device=0, verbose=False
        )[0]

        pred_img = img.copy()

        for box in pred_result.boxes:

            if int(box.cls[0]) != cls_id:
                continue

            x1, y1, x2, y2 = map(
                int, box.xyxy[0].cpu().tolist()
            )
            conf = float(box.conf[0])

            cv2.rectangle(
                pred_img, (x1, y1), (x2, y2),
                (0, 0, 255), 10
            )

            cv2.putText(
                pred_img,
                f"{class_names[cls_id]} {conf:.2f}",
                (x1, max(y1-15, 35)),
                cv2.FONT_HERSHEY_SIMPLEX,
                4, (0, 0, 255), 4
            )

        # Display
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        axes[0].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB))
        axes[1].set_title("YOLO Prediction")
        axes[1].axis("off")

        plt.suptitle(
            f"{class_names[cls_id]} | AP: {per_class_aps[cls_id]:.4f}"
        )

        plt.tight_layout()
        plt.show()

        break

## Bottom-3 Classes by mAP:


In [ ]:
import cv2
import glob
import os
import matplotlib.pyplot as plt

# Bottom-3 classes by mAP
bottom3_indices = sorted(
    range(len(per_class_aps)),
    key=lambda i: per_class_aps[i]
)[:3]

# Test images
test_images = glob.glob(
    os.path.join(dataset.location, "test", "images", "*.jpg")
)

print("🔴 Bottom-3 Classes by mAP:")

for cls_id in bottom3_indices:

    print(f"\nClass: {class_names[cls_id]}")
    print(f"AP: {per_class_aps[cls_id]:.4f}")

    # Find image containing this class
    for img_path in test_images:

        label_path = img_path.replace(
            os.sep + "images" + os.sep,
            os.sep + "labels" + os.sep
        )
        label_path = os.path.splitext(label_path)[0] + ".txt"

        if not os.path.exists(label_path):
            continue

        with open(label_path, "r") as f:
            labels = [
                list(map(float, line.split()))
                for line in f if line.strip()
            ]

        # Check whether this image contains the class
        if not any(int(label[0]) == cls_id for label in labels):
            continue

        # Read original image
        img = cv2.imread(img_path)
        h, w = img.shape[:2]

        # Ground Truth image
        gt_img = img.copy()

        for label in labels:
            cls, x, y, bw, bh = label

            if int(cls) != cls_id:
                continue

            x1 = int((x - bw/2) * w)
            y1 = int((y - bh/2) * h)
            x2 = int((x + bw/2) * w)
            y2 = int((y + bh/2) * h)

            cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(
                gt_img, class_names[cls_id],
                (x1, max(y1-8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 0), 2
            )

        # Model prediction
        pred_result = model.predict(
            img_path, device=0, verbose=False
        )[0]

        pred_img = img.copy()

        for box in pred_result.boxes:
            if int(box.cls[0]) != cls_id:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().tolist())
            conf = float(box.conf[0])

            cv2.rectangle(
                pred_img, (x1, y1), (x2, y2),
                (0, 0, 255), 12
            )

            cv2.putText(
                pred_img,
                f"{class_names[cls_id]} {conf:.2f}",
                (x1, max(y1-15, 35)),
                cv2.FONT_HERSHEY_SIMPLEX,
                2, (0, 0, 255), 4
            )

        # Display comparison
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        axes[0].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB))
        axes[1].set_title("YOLO Prediction")
        axes[1].axis("off")

        plt.suptitle(
            f"{class_names[cls_id]} | AP: {per_class_aps[cls_id]:.4f}"
        )

        plt.tight_layout()
        plt.show()

        break


# Comparative Analysis

In [ ]:
import matplotlib.pyplot as plt

precisions = results.box.p  
recalls = results.box.r     

# PR Curve
plt.figure(figsize=(10, 6))
for i in top3_indices:
    plt.plot([0, recalls[i], 1], [0, precisions[i], 0], label=f"{class_names[i]}")

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Top-3 Classes')
plt.legend()
plt.grid(True)
plt.show()

## Precision-Recall Curve for Overall Test Dataset

In [ ]:
import matplotlib.pyplot as plt
precision = results.box.mp
recall = results.box.mr

print(f"Overall Precision: {precision:.4f}")
print(f"Overall Recall: {recall:.4f}")

precision = results.box.mp
recall = results.box.mr

metrics = ["Precision", "Recall"]
values = [precision, recall]



In [ ]:
import matplotlib.pyplot as plt

precision = results.box.mp
recall = results.box.mr

plt.figure(figsize=(10, 6))

plt.plot(
    [0, recall, 1],
    [0, precision, 0],
    label="Overall Test Dataset"
)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for Overall Test Dataset')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from IPython.display import Image, display

confusion_matrix_path = os.path.join(
    str(results.save_dir),
    "confusion_matrix.png"
)

display(Image(filename=confusion_matrix_path))

In [ ]:
normalized_path = os.path.join(
    str(results.save_dir),
    "confusion_matrix_normalized.png"
)

display(Image(filename=normalized_path))

## Top-5 best detection index (highest confidence) and Bottom-5 worst detection index (lowest confidence)

In [ ]:
import glob
import os
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

# Load trained model
model = YOLO(str(best_weight_path))

# Test image paths
test_images = glob.glob(
    os.path.join(dataset.location, "test", "images", "*.jpg")
)

# Store detections
detections = []

for img_path in test_images:
    results_pred = model.predict(img_path, device=0, verbose=False)

    for box in results_pred[0].boxes:
        conf = float(box.conf[0])

        detections.append({
            "image": img_path,
            "confidence": conf,
            "class": int(box.cls[0])
        })

# Sort by confidence
detections.sort(key=lambda x: x["confidence"], reverse=True)

# Top 5 Best
print("\n🟢 Top 5 Best Detections:")

for item in detections[:5]:
    print(f"Class: {model.names[item['class']]}")
    print(f"Confidence: {item['confidence']:.4f}")
    display(Image.open(item["image"]))

# Top 5 Worst
print("\n🔴 Top 5 Worst Detections:")

for item in detections[-5:][::-1]:
    print(f"Class: {model.names[item['class']]}")
    print(f"Confidence: {item['confidence']:.4f}")
    display(Image.open(item["image"]))

In [ ]:

print("Evaluation folder:", results.save_dir)

for file in os.listdir(results.save_dir):
    print(file)

## F1-Confidence Curve

In [ ]:
from IPython.display import Image, display
f1_curve_path = os.path.join(
    str(results.save_dir),
    "BoxF1_curve.png"
)

display(Image(filename=f1_curve_path))

In [ ]:
from ultralytics.utils.plotting import save_one_box
import os

os.makedirs("best_detections", exist_ok=True)
os.makedirs("worst_detections", exist_ok=True)